In [ ]:
#=============================
# 1. Drive 마운트, 설치, 경로 세팅
#===============================
from google.colab import drive
drive.mount('/content/drive')

!pip -q install timm scikit-learn

In [ ]:
import os
import shutil
from pathlib import Path
from PIL import Image
import pandas as pd
from tqdm import tqdm

In [ ]:
# =========================================================
# 0) 설정
# =========================================================
# 전처리된 이미지 폴더
PREPROCESSED_BASE = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Preprocessed"
SPLITS = ["train", "val", "test"]

# 라벨 CSV (Training_Labels / Validation_Labels / Testing_Labels)
LABEL_TRAIN_CSV = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/csv/Training_Labels.csv"
LABEL_VAL_CSV   = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/csv/Validation_Labels.csv"
LABEL_TEST_CSV  = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/csv/Testing_Labels.csv"

LABEL_CSV_MAP = {
    "train": LABEL_TRAIN_CSV,
    "val": LABEL_VAL_CSV,
    "test": LABEL_TEST_CSV,
}

# 최종 데이터셋 출력 폴더
FINAL_BASE = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Final_Binary_DR"
QUARANTINE_BASE = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Quarantine_Binary_DR"

# 허용 이미지 확장자
IMG_EXTS = {".png", ".jpg", ".jpeg"}

# 채널 기준: 최종은 RGB로 통일 (L/LA/RGBA 등은 RGB 변환 시도)
FORCE_RGB = True


In [ ]:
# =========================================================
# 1) 유틸 함수
# =========================================================
def safe_mkdir(p: str):
    os.makedirs(p, exist_ok=True)

def find_id_col(df: pd.DataFrame) -> str:
    """RFMiD 라벨 CSV에서 이미지 ID 컬럼 자동 탐색"""
    candidates = ["ID", "id", "image_id", "ImageID", "img_id"]
    for c in candidates:
        if c in df.columns:
            return c
    # 그래도 없으면 첫 컬럼을 ID로 가정(많이 이런 형태임)
    return df.columns[0]

def find_dr_col(df: pd.DataFrame) -> str:
    """DR 라벨 컬럼 자동 탐색(정확히 'DR'인 경우가 대부분)"""
    # 가장 흔한 경우
    if "DR" in df.columns:
        return "DR"
    # 대소문자 무시 탐색
    for c in df.columns:
        if str(c).strip().lower() == "dr":
            return c
    raise ValueError(f"❌ DR 컬럼을 찾지 못했어. columns={list(df.columns)}")

def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMG_EXTS

def check_and_prepare_image(src_path: Path):
    """
    이미지 열기/검증 + (옵션) RGB 변환 + 해상도 체크
    반환: (ok: bool, reason: str, mode: str, size: (w,h), img_obj or None)
    """
    try:
        with Image.open(src_path) as img:
            img.verify()
        # verify 이후 재-open
        img = Image.open(src_path)
        mode = img.mode
        w, h = img.size

    # 같은 파일명 충돌 방지
    if dest.exists():
        dest = q_dir / f"{src_path.stem}__dup{src_path.suffix}"
    shutil.move(str(src_path), str(dest))
    return str(dest)

def save_final_image(img: Image.Image, dest_path: Path):
    safe_mkdir(str(dest_path.parent))
    # png로 저장(원본 확장자 유지하고 싶으면 dest_path.suffix에 맞춰 저장)
    img.save(dest_path)

In [ ]:
# =========================================================
# 2) 최종 폴더 구조 생성
#    - ImageFolder로 바로 학습 가능하게 split/0, split/1 로 구성
# =========================================================
for split in SPLITS:
    safe_mkdir(str(Path(FINAL_BASE) / split / "0"))
    safe_mkdir(str(Path(FINAL_BASE) / split / "1"))
    safe_mkdir(str(Path(QUARANTINE_BASE) / split))

In [ ]:
# =========================================================
# 3) split별 라벨 CSV 로드 → (이미지ID -> binary label) 맵 만들기
# =========================================================
label_map_by_split = {}
for split in SPLITS:
    csv_path = LABEL_CSV_MAP[split]
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"❌ 라벨 CSV가 없음: {csv_path}")

    df = pd.read_csv(csv_path)
    id_col = find_id_col(df)
    dr_col = find_dr_col(df)

    # ID를 문자열로 통일 (파일 stem과 비교 쉬움)
    df[id_col] = df[id_col].astype(str).str.strip()

    # DR 컬럼이 0/1이라고 가정 (RFMiD는 보통 0/1)
    # 혹시 'Yes/No' 등인 경우를 대비해 변환 로직 추가
    def to_bin(x):
        if pd.isna(x): return 0
        if isinstance(x, str):
            x2 = x.strip().lower()
            if x2 in ["1", "true", "yes", "y"]: return 1
            if x2 in ["0", "false", "no", "n"]: return 0
        try:
            return 1 if float(x) >= 1 else 0
        except:
            return 0

    df["DR_BIN"] = df[dr_col].apply(to_bin).astype(int)

    # { "1916": 0/1 } 형태로 맵핑
    label_map = dict(zip(df[id_col], df["DR_BIN"]))
    label_map_by_split[split] = label_map

    print(f"[{split}] rows={len(df)}  id_col={id_col}  dr_col={dr_col}  DR_pos={df['DR_BIN'].sum()}")

In [ ]:
# =========================================================
# 4) 본 작업: 이미지 검사 → 라벨 매칭 → 최종 폴더에 저장
# =========================================================
all_reports = []

for split in SPLITS:
    src_dir = Path(PREPROCESSED_BASE) / split
    if not src_dir.exists():
        raise FileNotFoundError(f"❌ 전처리 이미지 폴더가 없음: {src_dir}")

    img_paths = [p for p in src_dir.iterdir() if p.is_file() and is_image_file(p)]
    print(f"\n[{split}] found images:", len(img_paths))

    label_map = label_map_by_split[split]

    manifest_rows = []
    bad_count = 0

    for p in tqdm(img_paths, desc=f"Cleaning {split}"):
        img_id = p.stem  # 1916.png -> "1916"
        ok, reason, mode, size, img = check_and_prepare_image(p)



        # 1) 라벨 매칭
        if img_id not in label_map:
            # 라벨 없으면 quarantine 이동
            img.close()
            qpath = move_to_quarantine(p, split, "missing_label")
            all_reports.append({"split": split, "path": str(p), "status": "quarantine", "reason": "missing_label", "quarantine_path": qpath})
            bad_count += 1
            continue

        y = int(label_map[img_id])  # 0/1

        # 2) 최종 저장 위치 (split/0 or split/1)
        dest_path = Path(FINAL_BASE) / split / str(y) / f"{img_id}.png"

        manifest_rows.append({
            "split": split,
            "img_id": img_id,
            "label": y,
            "final_path": str(dest_path),
            "orig_path": str(p),
            "mode": mode,
            "width": size[0],
            "height": size[1],
        })
        all_reports.append({"split": split, "path": str(p), "status": "kept", "reason": "ok", "final_path": str(dest_path)})

    # split별 manifest 저장
    manifest_df = pd.DataFrame(manifest_rows)
    safe_mkdir(FINAL_BASE)
    manifest_csv_path = Path(FINAL_BASE) / f"manifest_{split}.csv"
    manifest_df.to_csv(manifest_csv_path, index=False)

    print(f"[{split}] kept={len(manifest_rows)}  quarantined={bad_count}")
    print(f"→ manifest saved: {manifest_csv_path}")

# 전체 리포트 저장
report_df = pd.DataFrame(all_reports)
safe_mkdir("/content/reports")
report_path = "/content/reports/data_cleaning_report.csv"
report_df.to_csv(report_path, index=False)

print("\n✅ DONE")
print("FINAL_BASE      =", FINAL_BASE)
print("QUARANTINE_BASE =", QUARANTINE_BASE)
print("REPORT          =", report_path)